# Market regime detection with a Gaussian HMM

Runs the full pipeline from `src/` for one market. Set the ticker and exposure map in the settings cell, then run all. Tables are saved to `results/` and figures to `figures/`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image
from scipy.optimize import brentq

from src.data import load_prices, emissions, clean_ticker, FEATURE_COLS
from src.hmm import select_n_states
from src.backtest import metrics, apply_costs, sweep_exposure_maps
from src.crosscheck import fit_markov_switching, ranked_regime_path, agreement
from src.ticker_pipeline import hmm_pipeline, HEADLINE_MAP

# hmmlearn raises a numpy deprecation warning on every fit
warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.options.display.float_format = '{:.3f}'.format

In [ ]:
TICKER = '^GSPC'        # any Yahoo Finance ticker, e.g. 'TLT', '^N225'
MAP_NAME = 'headline'   # 'headline' cuts exposure in crisis, 'mirror' raises it

# both maps fixed before any non-S&P run, so neither is tuned to a market's results
MAPS = {'headline': HEADLINE_MAP, 'mirror': {0: 0.0, 1: 0.5, 2: 1.0}}
EXPOSURE_MAP = MAPS[MAP_NAME]
NAME = clean_ticker(TICKER)

## 1. Data and features

In [ ]:
prices = load_prices(TICKER)
X, dates, next_day_ret = emissions(prices)
print(X.shape, dates.min().date(), '->', dates.max().date())

In [ ]:
# data checks: stale prices, impossible bars (0.01% slack for rounding in adjusted prices), extreme moves
close = prices['Close']
flat = close.diff() == 0
print('longest run of unchanged closes:', flat.groupby((~flat).cumsum()).sum().max())
print('closes outside High-Low:', int(((close > prices['High'] * 1.0001) | (close < prices['Low'] * 0.9999)).sum()))
close.pct_change().abs().nlargest(5)

## 2. Number of states

Fit on data before 2019, score BIC and log-likelihood on 2019 onward. On the S&P 500 both keep improving through 5 states, so there is no flat spot to pick; 3 is chosen because the gains decelerate and states beyond 3 only split the calm regime without a distinct interpretation. Every market uses the same 3 states so the results stay comparable. The 2019+ likelihood overlaps the walk-forward window, so this choice is not fully out of sample.

In [ ]:
train_end = int((dates < '2019-01-01').sum())
pd.DataFrame(select_n_states(X, train_end)).set_index('n')

## 3. Full-sample fit (interpretation only)

Viterbi path, state means, transition matrix, implied vs realised dwell times.

In [ ]:
# figures are saved for the headline map only, so a mirror run doesn't overwrite them
out = hmm_pipeline(TICKER, exposure_map=EXPOSURE_MAP, plots=MAP_NAME == 'headline')
states, results, ranked = out['states'], out['results'], out['ranked']
out['state_table']

In [ ]:
Image(f'figures/{NAME}_regime_price.png')

In [ ]:
Image(f'figures/{NAME}_transition_heatmap.png')

## 4. Walk-forward overlay vs baselines

Expanding window (`min_train=1000`, refit every 21 days), forward-filtered states. The fit takes about 6 minutes per market, so the states are cached in `data/` under the ticker and settings.

In [ ]:
print(states.index.min().date(), '->', states.index.max().date(), len(states), 'days')
out['regime_share']

In [ ]:
out['table']

In [ ]:
Image(f'figures/{NAME}_equity_curves.png')

In [ ]:
Image(f'figures/{NAME}_drawdowns.png')

## 5. Exposure maps and transaction costs

In [ ]:
maps = {'1.0 / 0.5 / 0.0 (headline)': HEADLINE_MAP,
        '1.0 / 0.6 / 0.2': {0: 1.0, 1: 0.6, 2: 0.2},
        '1.0 / 0.75 / 0.25': {0: 1.0, 1: 0.75, 2: 0.25},
        '1.0 / 1.0 / 0.0 (crisis only)': {0: 1.0, 1: 1.0, 2: 0.0}}
sweep_exposure_maps(states, next_day_ret, maps, cost_bps=5)

In [ ]:
# net Sharpe of every strategy at several per-trade cost assumptions
bps_grid = [0, 2, 5, 10]
net_sharpe = pd.DataFrame({
    name: [metrics(apply_costs(f['strat_ret'], f['exposure'], b), f['exposure'])['sharpe'] for b in bps_grid]
    for name, f in results.items()}, index=[f'{b} bps' for b in bps_grid])
net_sharpe

In [ ]:
# cost at which the overlay's net Sharpe falls to the vol-target's
def gap(bps):
    o, v = results['regime overlay'], results['vol target']
    return (metrics(apply_costs(o['strat_ret'], o['exposure'], bps), o['exposure'])['sharpe']
            - metrics(apply_costs(v['strat_ret'], v['exposure'], bps), v['exposure'])['sharpe'])

if gap(0) > 0 > gap(500):
    print(f'break-even cost: {brentq(gap, 0, 500):.1f} bps per unit of exposure traded')
else:
    print('no break-even in 0-500 bps: gap at 0 bps =', round(gap(0), 3), '| at 500 bps =', round(gap(500), 3))

## 6. Cross-check: Markov-switching regression

`MarkovRegression` on the 100x log return only (switching mean and variance), against the HMM's full-sample Viterbi path.

In [ ]:
ms = fit_markov_switching(pd.Series(X[:, FEATURE_COLS.index('Return')], index=dates), k_regimes=3)
sigma2 = np.array([ms.params[f'sigma2[{i}]'] for i in range(3)])
const = np.array([ms.params[f'const[{i}]'] for i in range(3)])
o = np.argsort(sigma2)
pd.DataFrame({'mean_ret': const[o], 'vol': np.sqrt(sigma2[o]), 'expected_dwell': np.asarray(ms.expected_durations)[o]},
             index=['calm', 'choppy', 'crisis'])

In [ ]:
ms_path = ranked_regime_path(ms, dates)
ag = agreement(pd.Series(ranked, index=dates), ms_path)
print(f"same-day agreement: {ag['same_day_pct']:.1f}%  |  crisis-day Jaccard: {ag['crisis_jaccard']:.2f}")
ag['confusion']

## 7. Save results

One CSV per market and map in `results/`: the metrics table plus Sharpe net of 5 bps. Written findings are in the README.

In [ ]:
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)
summary = out['table'].copy()
summary['sharpe_5bps'] = net_sharpe.loc['5 bps']
summary['map'] = MAP_NAME
summary.to_csv(RESULTS_DIR / f'{NAME}_{MAP_NAME}.csv', index_label='strategy', float_format='%.4f')
summary